・Lightgbmシングルモデルによる、House Pricesコンペのソリューション

Lightgbmのシングルモデルで取り組んだのは、前処理をする手間をかけずに特徴量が効いているかどうか簡単に実験できるから。

詳しくはu++さんの以下の記事を参照
https://upura.hatenablog.com/entry/2019/10/29/184617


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd
from scipy.stats import skew
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn import preprocessing
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sub = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

train, test, submissionファイルのデータの確認

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
sub.head()

In [ ]:
n_train = len(train) # trainデータの長さ 
features = pd.concat([train, test], sort=False).reset_index(drop=True)

In [ ]:
"""
objects = []
for i in features.columns:
    if features[i].dtype == object:
        objects.append(i)
features.update(features[objects].fillna('None'))

features['LotFrontage'] = features.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numerics = []
for i in features.columns:
    if features[i].dtype in numeric_dtypes:
        numerics.append(i)
features.update(features[numerics].fillna(0))
"""

In [ ]:
"""
numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numerics2 = []
for i in features.columns:
    if features[i].dtype in numeric_dtypes:
        numerics2.append(i)
skew_features = features[numerics2].apply(lambda x: skew(x)).sort_values(ascending=False)
print(skew_features)

high_skew = skew_features[skew_features > 0.5]
skew_index = high_skew.index

for i in skew_index:
    features[i] = boxcox1p(features[i], boxcox_normmax(features[i] + 1))
"""


In [ ]:
#　ここに特徴量を加える
"""
features['YrBltAndRemod']=features['YearBuilt']+features['YearRemodAdd']
features['TotalSF']=features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']

features['Total_sqr_footage'] = (features['BsmtFinSF1'] + features['BsmtFinSF2'] +
                                 features['1stFlrSF'] + features['2ndFlrSF'])

features['Total_Bathrooms'] = (features['FullBath'] + (0.5 * features['HalfBath']) +
                               features['BsmtFullBath'] + (0.5 * features['BsmtHalfBath']))

features['Total_porch_sf'] = (features['OpenPorchSF'] + features['3SsnPorch'] +
                              features['EnclosedPorch'] + features['ScreenPorch'] +
                              features['WoodDeckSF'])
features['haspool'] = features['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
features['has2ndfloor'] = features['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasgarage'] = features['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
features['hasbsmt'] = features['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasfireplace'] = features['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)
"""


In [ ]:
test = features[n_train:].reset_index(drop=True) #結合されているデータをtrainとtestに再分割
train = features[:n_train]

In [ ]:
train.info()

In [ ]:
def label_encoding(train: pd.DataFrame, test: pd.DataFrame, col_definition: list):
    n_train = len(train) # trainデータの長さ 
    train = pd.concat([train, test], sort=False).reset_index(drop=True) # trainとtestデータを結合
    for column in col_definition: # col_definitionにカテゴリ変数のカラム名のリストが入る columnは各カテゴリ名がstring型で入る
        try:
            lbl = preprocessing.LabelEncoder()
            train[column] = lbl.fit_transform(list(train[column].values)) # LabelEncodingを適用
        except:
            print(column)
    test = train[n_train:].reset_index(drop=True) #結合されているデータをtrainとtestに再分割
    train = train[:n_train]
    return train, test

カテゴリ変数にラベルエンコーディングを適用

In [ ]:
categorical_cols = list(train.select_dtypes(include=['object']).columns) # カテゴリ変数(type=object)が入っているカラムの名前をリストにする
print(categorical_cols)
# ラベルエンコーディング
train, test = label_encoding(train, test, categorical_cols)

In [ ]:
X = train.drop(['Id', 'SalePrice'], axis=1)
y = np.log1p(train['SalePrice'])
X_test = test.drop(['Id','SalePrice'], axis=1)

In [ ]:
outliers = [30, 88, 462, 631, 1322]
X = X.drop(X.index[outliers])
X.reset_index(inplace=True, drop=True)
y = y.drop(y.index[outliers])
y.reset_index(inplace=True, drop=True)

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
y_preds = []
models = []
oof_train = np.zeros((len(X),))
cv = KFold(n_splits=5, shuffle=True, random_state=0)

params = {
    'num_leaves': 24,
    'max_depth': 6,
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05
}

for fold_id, (train_index, valid_index) in enumerate(cv.split(X)):
    X_tr = X.loc[train_index, :]
    X_val = X.loc[valid_index, :]
    y_tr = y[train_index]
    y_val = y[valid_index]

    lgb_train = lgb.Dataset(X_tr,
                            y_tr,
                            categorical_feature=categorical_cols)

    lgb_eval = lgb.Dataset(X_val,
                           y_val,
                           reference=lgb_train,
                           categorical_feature=categorical_cols)

    model = lgb.train(params,
                      lgb_train,
                      valid_sets=[lgb_train, lgb_eval],
                      verbose_eval=10,
                      num_boost_round=1000,
                      early_stopping_rounds=10)


    oof_train[valid_index] = model.predict(X_val,
                                           num_iteration=model.best_iteration)
    y_pred = model.predict(X_test,
                           num_iteration=model.best_iteration)

    y_preds.append(y_pred)
    models.append(model)



In [ ]:
print(f'CV: {np.sqrt(mean_squared_error(y, oof_train))}')

In [ ]:
y_pred = np.expm1(model.predict(X_test,num_iteration=model.best_iteration))

対数を取っていた予測値を元に戻す

In [ ]:
y_sub = sum(y_preds) / len(y_preds)
y_sub = np.expm1(y_sub)
sub['SalePrice'] = y_sub
sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
pd.set_option('display.max_rows', 100)
importance = pd.DataFrame(model.feature_importance(importance_type="gain"), index=X.columns, columns=['importance'])
importance = importance.sort_values("importance", ascending=False)
display(importance)

今後

・今回はLightGBMだけでモデルを作ったが、精度を上げるためにRidgeやSVM,XGBoostなどと組み合わせてアンサンブル学習するべき
・より良いバリデーション手法を考えることができる？